# Trustworthy SLM Agent — Phase 2: Fine-Tuning
This notebook fine-tunes `Qwen2.5-1.5B-Instruct` with QLoRA on the
"Why Language Models Hallucinate" Q&A dataset, using a free Colab T4 GPU.

**Before running:** Runtime → Change runtime type → T4 GPU → Save.

## 1. Confirm GPU access

In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets pyyaml huggingface_hub

## 3. Get your project files into Colab

**Option A (recommended): clone your GitHub repo directly.**
Replace the URL with your actual repo.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/trustworthy-slm-agent.git
%cd trustworthy-slm-agent

**Option B: if your repo is private and cloning fails**, instead upload just
the four files needed for training using the Colab file upload button (left
sidebar → Files → upload icon), placing them at:
- `configs/lora_config.yaml`
- `src/finetune.py`
- `data/qa_train.jsonl`
- `data/qa_val.jsonl`

Skip Option A's cell above if you do this instead.

## 4. Log in to Hugging Face (needed to push your adapter)

In [ ]:
from huggingface_hub import login
login()  # paste your HF write token when prompted

## 5. Edit your config's Hub repo_id before training

Open `configs/lora_config.yaml` in Colab's file browser (left sidebar) and
replace `YOUR_HF_USERNAME` under `hub: repo_id:` with your actual Hugging
Face username. Save the file, then continue.

## 6. Run fine-tuning

In [ ]:
!python src/finetune.py --config configs/lora_config.yaml

## 7. Quick qualitative check — compare base vs. fine-tuned outputs

Run a few sample questions through both the original base model and your
newly fine-tuned adapter, side by side.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "results/lora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, quantization_config=bnb_config, device_map="auto"
)
finetuned_model = PeftModel.from_pretrained(base_model, adapter_path)

test_questions = [
    "Why do language models hallucinate according to this paper?",
    "What is the singleton rate?",
    "Does RAG fully solve hallucination?",
]

def ask(model, question, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

system_prompt = (
    "You are a helpful assistant with expert knowledge of the paper "
    "'Why Language Models Hallucinate' by Kalai, Nachum, Vempala, and Zhang (2025)."
)

for q in test_questions:
    print("="*80)
    print("Q:", q)
    print("\n[FINE-TUNED]:", ask(finetuned_model, q, system_prompt))

## 8. Download your adapter (backup, in case Hub push failed)

In [ ]:
!zip -r lora_adapter.zip results/lora_adapter
from google.colab import files
files.download("lora_adapter.zip")